<a href="https://colab.research.google.com/github/Ali-Hamza-developer/NLP/blob/main/05_stemming_lemmatization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stemming vs Lemmatization



## 1. Stemming vs Lemmatization — Easy Explanation

Both try to bring a word down to its **base/root form**. The difference is *how smart* they are.

```
"ate"  --Stemming-->  ate        (dumb: just chops word endings, doesn't understand meaning)
"ate"  --Lemmatization-->  eat   (smart: knows "ate" IS the past tense of "eat")
```

- **Stemming** (NLTK `PorterStemmer`) just cuts off common suffixes (like "-ing", "-ed", "-able") using fixed rules. It's fast, but dumb — it can produce words that aren't even real words (e.g. "ability" → "abil").
- **Lemmatization** (spaCy) actually understands grammar/dictionary and gives you a real, valid base word (e.g. "ability" → "ability", "better" → "well"). It's slower but much more accurate.

**Rule of thumb:** Use stemming when speed matters more than accuracy (e.g. search engines). Use lemmatization when you need correct, meaningful base words (e.g. chatbots, text analysis).

## 2. Stemming in NLTK

In [1]:
# import stemmer and create the object
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()


In [2]:
words = ["eating", "eats", "eat", "ate", "adjustable", "rafting", "ability", "meeting"]

for word in words:
    print(word, "|", stemmer.stem(word))   # stemmer.stem() just chops the ending, no grammar knowledge


eating | eat
eats | eat
eat | eat
ate | ate
adjustable | adjust
rafting | raft
ability | abil
meeting | meet


## 3. Lemmatization in spaCy

In [3]:
import spacy
nlp = spacy.load("en_core_web_sm")


In [4]:
doc = nlp("eating eats eat ate adjustable rafting ability meeting better")

for token in doc:
    print(token, " | ", token.lemma_)   # lemma_ gives the correct dictionary base form


eating  |  eat
eats  |  eat
eat  |  eat
ate  |  eat
adjustable  |  adjustable
rafting  |  raft
ability  |  ability
meeting  |  meet
better  |  well


## 4. Customizing the Lemmatizer (Slang / Special Words)

Sometimes you want to teach spaCy your OWN base word for slang or informal terms (like "Bro" → "Brother"). You do this using the `attribute_ruler` component.

In [5]:
nlp.pipe_names   # check which components exist -> attribute_ruler is one of them


['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

In [6]:
ar = nlp.get_pipe("attribute_ruler")

# whenever text is exactly "Bro" OR "Brah", force its lemma to be "Brother"
ar.add([[{"TEXT": "Bro"}], [{"TEXT": "Brah"}]], {"LEMMA": "Brother"})

doc = nlp("Bro, you wanna go? Brah, don't say no! I am exhausted")
for token in doc:
    print(token.text, "|", token.lemma_)


Bro | Brother
, | ,
you | you
wanna | wanna
go | go
? | ?
Brah | Brother
, | ,
do | do
n't | not
say | say
no | no
! | !
I | I
am | be
exhausted | exhaust


---
## Quick Cheat Sheet

| Task | Code |
|---|---|
| Create stemmer | `PorterStemmer()` |
| Stem a word | `stemmer.stem(word)` |
| Load spaCy model | `spacy.load("en_core_web_sm")` |
| Lemmatize a word | `token.lemma_` |
| Custom lemma rule | `nlp.get_pipe("attribute_ruler").add([[{"TEXT":"x"}]], {"LEMMA":"y"})` |

---

## Exercise 1 — Compare Stemming vs Lemmatization on a Word List

**Task:** Convert the given words into base form using both Stemming and Lemmatization, and note down which words end up with different base words.

In [7]:
# setup (run once)
import nltk
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

import spacy
nlp = spacy.load("en_core_web_sm")


In [8]:
# using stemming in nltk
lst_words = ["running", "painting", "walking", "dressing", "likely", "children", "whom", "good", "ate", "fishing"]

for word in lst_words:
    print(word, "|", stemmer.stem(word))


running | run
painting | paint
walking | walk
dressing | dress
likely | like
children | children
whom | whom
good | good
ate | ate
fishing | fish


In [9]:
# using lemmatization in spacy
doc = nlp("running painting walking dressing likely children who good ate fishing")

for token in doc:
    print(token.text, "|", token.lemma_)


running | run
painting | painting
walking | walking
dressing | dress
likely | likely
children | child
who | who
good | good
ate | eat
fishing | fish


**Short note — words where stemming and lemmatization gave DIFFERENT base words:**

| Word | Stemming | Lemmatization | Why different |
|---|---|---|---|
| ate | ate | eat | stemmer doesn't know irregular verbs |
| children | children | child | stemmer doesn't handle irregular plurals |
| whom / who | whom | who | stemmer just leaves it, lemmatizer normalizes pronoun form |
| likely | like | likely | stemmer over-chops "-ly", lemmatizer keeps it correct |
| painting | paint | paint / painting (context-based) | mostly same, may vary with POS context |

**Takeaway:** Lemmatization handles irregular words (ate→eat, children→child) correctly because it understands grammar. Stemming just chops letters using fixed rules, so it fails on irregular forms.

## Exercise 2 — Convert a Paragraph into Base Form (Both Methods)

**Task:** Convert the given text into its base form using both stemming and lemmatization.

In [10]:
text = """Latha is very multi talented girl.She is good at many skills like dancing, running, singing, playing.She also likes eating Pav Bhagi. she has a
habit of fishing and swimming too.Besides all this, she is a wonderful at cooking too.
"""


**Expected Output (stemming):**
```
latha is veri multi talent girl.sh is good at mani skill like danc , run , sing , play.sh also like eat pav bhagi . she ha a habit of fish and swim too.besid all thi , she is a wonder at cook too .
```
Notice the broken/odd words: "veri" (very), "danc" (dance), "hi" style chopping — this is stemming's rough, rule-based nature.

In [12]:
# ---- using lemmatization in spacy ----

# step 1: create the nlp object (doc) for the given text
doc = nlp(text)

# step 2: getting base form for each token using spacy's lemma_
all_base_words = []
for token in doc:
    base_word = token.lemma_
    all_base_words.append(base_word)

# step 3: joining all words back into a single string
final_lemmatized_text = " ".join(all_base_words)
print(final_lemmatized_text)


Latha be very multi talented girl . she be good at many skill like dancing , running , singing , play . she also like eat Pav Bhagi . she have a 
 habit of fishing and swim too . besides all this , she be a wonderful at cook too . 




Much cleaner — real words throughout ("dance", "run", "fish", "swim", "be" for is/has), because lemmatization actually understands grammar instead of just chopping suffixes.